# Tutorial preTP1 - 2026

La idea de esta notebook es introducirles a las herramientas básicas para poder cargar, manipular y visualizar grafos usando Python.

Vamos a trabajar primero con NetworkX, una librería muy accesible, que ofrece muchas de las funcionalidades necesarias para grafos de forma intuitiva. Como contra, no es tan eficiente para resolver algunas tareas computacionalmente costosas.

## Importamos librerias

In [ ]:
# Herramientas de sistema para archivos
import os
from glob import glob
from pathlib import Path

# La librería de Grafos
import networkx as nx

# Números y Datos
import numpy as np
import pandas as pd

# Para visualización
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns

# Parámetros de visualización
sns.set_context('talk')
mpl.rcParams['figure.figsize'] = (8, 6)

## Los datos

Bajamos y revisamos el dataset

Utilizar el signo de exclamación permite correr comandos como si fuera la terminal de la computadora. Podemos utilizarlo para descargar (wget) o descomprimir (unzip) archivos, y también para instalar librerías de python (pip install) entre otras.

In [ ]:
!wget -q https://www.dropbox.com/s/eei6gnw10o20mcn/DataSujetos.zip?dl=1 -O DataSujetos.zip
!unzip -q -o DataSujetos
!wget -q -O aal_extended_withCoords.csv https://www.dropbox.com/scl/fi/pu1qdch4c3khk0uol9l0w/aal_extended_withCoords.csv?rlkey=bsnfot3b9ycp7slble59e1cws&dl=1

### Atlas de coordenadas
Son 116 regiones cerebrales
La columna "1" indica el nombre de la subregión, la columna "3" indica a qué región cerebral pertenece, la columna "4" en qué hemisferio se localiza (L=izquierda, R=derecham, --= está en la línea media), las columnas "5", "6" y "7" indican las coordenadas x, y, z de cada subregión

In [ ]:
region_names = pd.read_csv("aal_extended_withCoords.csv", header=None, index_col=0)
region_names.index.name = ""
region_names.columns = ["subregion", "COD", "region", "hemisferio", "X", "Y", "Z"]
region_names

### Ejercicio 1

- ¿Cuántas regiones hay en el dataset?
- ¿Cuantos nodos tiene cada región?
- ¿Entre que valores de coordenadas se encuentra cada región?

*PISTA* -> Esto sigue sin ser un grafo, podemos analizarlo usando las herramientas típicas de pandas (`value_counts`, `groupby`, `max/min`, etc).

In [ ]:
# ESCRIBIR EL CÓDIGO AQUI



## Lista de archivos
Tenemos varios sujetos para 4 condiciones, una correspondiente a vigilia y 3 a diferentes estadios del sueño no REM (*rapid eye movement*):<br>

"W": despierto ("Wake")
<br> "N1": sueño "leve"
<br>"N2": sueño "intermedio"
<br>"N3": sueño "profundo"

In [ ]:
DIR = Path('DataSujetos')
sorted([filename.name for filename in DIR.glob('*')])

Podemos usar la funcionalidad de `glob` para filtrar según distintos patrones. Por ejemplo, solo vemos los archivos que comienzan en "W_", correspondientes a la condición "despierto".

Pueden ver otras posibilidades en https://docs.python.org/es/3/library/pathlib.html#pathlib.Path.glob

In [ ]:
sorted([filename.name for filename in DIR.glob('W_*')])

## Vamos a analizar el sujeto 1 de la condición W
Levantamos los datos y los transformamos en una matriz de adyacencia pesada, donde le quitamos los valores de 1 de la diagonal (las auto correlaciones de cada nodo)

In [ ]:
filename = DIR / 'W_suj1.csv'
df = pd.read_csv(filename, header=None)

mat_adyac_pesada = df.values # Extraemos los valores de la matriz
n = mat_adyac_pesada.shape[0] # Obtenemos una dimensión (ambas son iguales)
mat_adyac_pesada -= np.diag(np.ones(n)) # Restamos una diagonal de valores 1 a la matriz

Graficamos la matriz de ayacencia pesada, utilizando el plot `heatmap` de seaborn

https://seaborn.pydata.org/generated/seaborn.heatmap.html

In [ ]:
sns.heatmap(mat_adyac_pesada)
plt.show()

Tenemos un problema con los colores que elije la función por defecto, ya que no reflejan claramente la presencia de correlaciones (positivas) y autocorrelaciones (negativas).

Lo correcto sería elegir un mapa de colores divergente

### Ejercicio 2

- Visualizar la matriz con una paleta de colores divergente. Pueden investigar opciones en https://seaborn.pydata.org/tutorial/color_palettes.html o
https://matplotlib.org/stable/users/explain/colors/colormaps.html

In [ ]:
cmap_divergente = ""  # COMPLETAR

ax = sns.heatmap(mat_adyac_pesada, cmap=cmap_divergente)
plt.show()

## Binarización

Le ponemos un umbral "th" (threshold) para binarizarla hacia una matriz de adyacencia no pesada.

Solo nos quedamos con los enlaces cuya correlación sea mayor o igual a este valor.

In [ ]:
th = 0.65 # Podemos variar este umbral (OJO, VUELVAN A 0.65 ANTES DE CONTINUAR!)
mat_adyac_no_pesada = mat_adyac_pesada >= th # Binarizamos utilizando una condicion booleana sobre el valor de los pesos.
# Recordemos que 1 es equivalente a True y 0 a False en Python

sns.heatmap(mat_adyac_no_pesada)
plt.show()

### Ejercicio 3

Estamos quedandonos con los valores que sean mayores o iguales que un umbral positivo. Sin embargo, tenemos una matriz con valores de correlación tanto positivos como negativos, por lo que podríamos querer mantener los enlaces que tengan un cierto nivel de correlación tanto positiva como negativa.


- Cambiar la binarización para quedarse con los enlaces cuyo VALOR ABSOLUTO sea mayor o igual al umbral.


PISTA -> Numpy es su amigo!

In [ ]:
th = 0.65
mat_adyac_no_pesada_abs = mat_adyac_pesada >= th # MODIFICAR ESTA PARTE

sns.heatmap(mat_adyac_no_pesada_abs, cbar=False)
plt.show()

## NetworkX

Creamos un grafo con la librería Networkx a partir de la matriz de Numpy. Para convertir desde otros tipos de datos pueden buscar aca:
https://networkx.org/documentation/stable/reference/convert.html

Pueden investigar otras funcionalidades para cargar grafos desde diversos formatos acá:
https://networkx.org/documentation/stable/reference/readwrite/index.html

In [ ]:
G = nx.from_numpy_array(mat_adyac_no_pesada)

### Ejercicio 4.1

- ¿Qué tipo de objeto es el grafo `G`?

In [ ]:
# ESCRIBIR CÓDIGO AQUÍ


Vemos si se trata de un grafo conectado.
Para esto podemos utilizar una función auxiliar de la librería y pasarle el objeto del grafo.

In [ ]:
print('Es conectado?:', nx.is_connected(G))

###  Ejercicio 4.2

Responder estas preguntas usando funciones de NetworkX. **Recomendamos buscar en la documentación antes de consultar a su IA de confianza.**

- ¿Es un grafo dirigido? ¿Es pesado?
- ¿Qué cantidad de nodos y enlaces tiene?

In [ ]:
# ESCRIBIR CÓDIGO AQUÍ


## Visualizaciones

Graficamos nuestro grafo con los nodos dispuestos de diferentes formas

In [ ]:
plt.figure(figsize=(18, 6))

plt.subplot(131)
plt.title('Sin especificar posiciones', fontsize=14)
nx.draw(G, with_labels=True, font_weight='bold')

plt.subplot(132)
plt.title('Con nodos a lo largo de un círculo', fontsize=14)
layout_circle = nx.circular_layout(G)
nx.draw(G, layout_circle, with_labels=True, font_weight='bold')

plt.subplot(133)
plt.title('Con nodos en sus coordenadas cerebrales', fontsize=14)
layout_coords = dict(zip(range(n),np.array([region_names["Y"].values, region_names["Z"].values]).T))
nx.draw(G, layout_coords, with_labels=True, linewidths=1)

plt.show()

### Ejercicio 5

- Observar los distintos objetos de layout. ¿Cómo estan construidos los diccionarios?
- Graficar utilizando las otras combinaciones posibles de coordenadas cerebrales

In [ ]:
# ESCRIBIR EL CÓDIGO AQUÍ


## Visualizando características

Podemos visualizar los nodos de acuerdo a sus características. Por ejemplo, su grado.

Antes de eso, obtenemos el listado de nodos y sus grados.

(¿Que tipo de objetos devuelven?)

In [ ]:
G.nodes() # Nodos del grafo G

In [ ]:
G.degree() # Grado de cada nodo del grafo G

Con estos valores construimos el layout y un listado de colores en función del grado.

Vamos a dibujar también cada componente (enlaces y nodos) por separado, lo que permite tener mayor control

In [ ]:
layout = dict(zip(
    G.nodes(),
    np.column_stack([region_names["Y"].values, region_names["Z"].values])
))

node_color = [d for _, d in G.degree()]

plt.figure(figsize=(9, 6))
plt.title('Con nodos coloreados de acuerdo a su grado', fontsize=14)

# dibujar edges y nodos por separado
nx.draw_networkx_edges(G, layout, width=.5, edge_color="#5c5b5a")

nodes = nx.draw_networkx_nodes(
    G,
    layout,
    node_size=30,
    node_color=node_color,
    cmap=plt.cm.Blues,
)

plt.colorbar(nodes, ax=plt.gca())

plt.show()

### Ejercicio 6

- Visualizar los nodos de acuerdo a su coeficiente de clustering.

PISTA -> Se utiliza una función de la librería, no un método del grafo (como en degree)

PISTA 2 -> Al devolver un diccionario, hay que iterarlo usando `.items()` (a diferencia del grado donde podemos iterar directamente sobre lo que devuelve `G.degree`)

In [ ]:
# Codigo


## ¿Cuál es el nodo con más enlaces? (Grado mas alto)

In [ ]:
sorted_nodes = sorted(G.degree, key=lambda x: x[1], reverse=True)
print(f'El nodo con mayor grado es el {sorted_nodes[0][0]}, que posee {sorted_nodes[0][1]} enlaces' )

## Comparando dos estados

Cargamos para un **mismo sujeto**, dos estados distintos.

In [ ]:
filename = DIR / 'W_suj1.csv'
df = pd.read_csv(filename, header=None)
mat_adyac_pesada_wake = df.values
n = mat_adyac_pesada_wake.shape[0]
mat_adyac_pesada_wake -= np.diag(np.ones(n))

filename = DIR / 'N3_suj1.csv'
df = pd.read_csv(filename, header=None)
mat_adyac_pesada_sleep = df.values
n = mat_adyac_pesada_sleep.shape[0]
mat_adyac_pesada_sleep -= np.diag(np.ones(n))

Graficamos ambas matrices de adyacencia. ¿Qué se observa?

(Recuerden cambiar a un cmap adecuado)

In [ ]:
cmap = "rocket"

In [ ]:
plt.figure(figsize=(12,4))

plt.subplot(121)
sns.heatmap(mat_adyac_pesada_wake, cmap=cmap)
plt.title('Sujeto despierto')

plt.subplot(122)
sns.heatmap(mat_adyac_pesada_sleep, cmap=cmap)
plt.title('Sujeto dormido')

plt.show()

Tenemos un primer problema, ya que ambas redes tienen rangos de pesos diferentes. Cambiemos la escala para ver mejor.

In [ ]:
vmin = np.min([mat_adyac_pesada_wake.min(), mat_adyac_pesada_sleep.min()])
vmax = np.max([mat_adyac_pesada_wake.max(), mat_adyac_pesada_sleep.max()])

plt.figure(figsize=(12,4))

plt.subplot(121)
sns.heatmap(mat_adyac_pesada_wake, cmap=cmap, vmin=vmin, vmax=vmax)
plt.title('Sujeto despierto')

plt.subplot(122)
sns.heatmap(mat_adyac_pesada_sleep, cmap=cmap, vmin=vmin, vmax=vmax)
plt.title('Sujeto dormido')

plt.show()

## Binarización por densidad

Para poder hacer mejores comparaciones, vamos a binarizar ambas redes pero no usando un umbral fijo, sino buscando que ambas tengan la **misma densidad*

MUY ÚTIL: Definamos una función para binarizar una matriz de adyacencia pesada no dirigida en función de una densidad deseada

In [ ]:
def density_to_th(weights, target_density):
    n = weights.shape[0]
    tril_idx = np.tril_indices(n, -1) # Buscamos los indices del triangulo inferior, ignorando la diagonal
    c = sorted(np.array(list(weights[tril_idx].reshape(-1))), reverse=True) # Ordenamos los valores
    return c[int((len(c) - 1) * target_density)] # Devolvemos el valor de corte necesario para obtener la densidad objetivo

Y en caso de querer trabajar con los valores absolutos, tenemos que modificarla levemente

In [ ]:
def density_to_th_abs(weights, target_density):
    n = weights.shape[0]
    tril_idx = np.tril_indices(n, -1)
    c = sorted(np.abs(np.array(list(weights[tril_idx].reshape(-1))), reverse=True))
    return c[int((len(c) - 1) * target_density)]

Vamos a trabajar con la primera versión, buscando una densidad de 10%

In [ ]:
densidad = 0.1 # comparemos ambos estados con una densidad de enlaces de 10%

plt.figure(figsize=(12, 6))

plt.subplot(121)
thWake = density_to_th(mat_adyac_pesada_wake, densidad)
sns.heatmap(mat_adyac_pesada_wake >= thWake)
plt.title(f'Despierto, binarizado a {thWake}')

plt.subplot(122)
thSleep = density_to_th(mat_adyac_pesada_sleep, densidad)
sns.heatmap(mat_adyac_pesada_sleep >= thSleep)
plt.title(f'Dormido, binarizado a {thSleep}')

plt.show()

### Ejercicio 7

- Siendo que los enlaces representan situaciones donde dos regiones del cerebro (nodos) se activan de forma correlacionada, ¿qué conclusión pueden sacar de la diferencia entre los estados dormido y despierto?

- ¿Ocurre algo distinto si comparan el estado despierto con otro de los dos momentos del sueño no REM?

## Distribuciones de grado

Usando un histograma, veamos como son las distribuciones de grado en ambos estados

In [ ]:
plt.figure(figsize=(18, 4))

plt.subplot(121)
G_sleep = nx.from_numpy_array(mat_adyac_pesada_sleep >= thSleep)
degrees_sleep = [G_sleep.degree(n) for n in G_sleep.nodes()]
hist_sleep, bins_sleep, _ = plt.hist(degrees_sleep, bins=15, density=True)
plt.xlabel('k')
plt.ylabel('p(k)')
plt.title('Distribucion de grados, durmiendo')

plt.subplot(122)
G_wake = nx.from_numpy_array(mat_adyac_pesada_wake >= thWake)
degrees_wake = [G_wake.degree(n) for n in G_wake.nodes()]
hist_wake, bins_wake, _ = plt.hist(degrees_wake, bins=15, density=True)
plt.xlabel('k')
plt.ylabel('p(k)')
plt.title('Distribucion de grados, despierto')

plt.show()

¿Qué diferencias observan?

## Escala log-log

In [ ]:
def plot_degree_distribution(degrees, title):
    degrees = np.array(degrees)

    # filtrar grados > 0
    degrees = degrees[degrees > 0]

    # bins logarítmicos
    bins = np.logspace(
        np.log10(degrees.min()),
        np.log10(degrees.max()),
        30
    )

    # histograma
    hist, bin_edges = np.histogram(degrees, bins=bins, density=True)

    # centros de bins
    x = (bin_edges[:-1] + bin_edges[1:]) / 2

    # plot
    plt.plot(x, hist, 'o')
    plt.xscale('log')
    plt.yscale('log')
    plt.xlabel('k')
    plt.ylabel('p(k)')
    plt.title(title)


plt.figure(figsize=(18,6))

plt.subplot(121)
plot_degree_distribution(degrees_sleep, 'Distribución de grados (sleep)')

plt.subplot(122)
plot_degree_distribution(degrees_wake, 'Distribución de grados (wake)')

plt.show()